# Практическая работа №8. Перенос базы из SQLite в PostgreSQL и работа в DBeaver
## Конспект пары для преподавателя (по порядку проведения занятия)

### План пары (90 минут)
| Время | Этап пары | Действия преподавателя (база `club_demo`) | Действия студентов (базы `club_var01..20`) |
| :--- | :--- | :--- | :--- |
| **До пары** | **Этап 0. Подготовка** | Запускает ячейку ниже для заливки `demo/variant_00` в базу `club_demo`. | — |
| **00–15 мин** | **Этап 1. Архитектура и DBeaver** | Показывает отличие сервера PostgreSQL от файла SQLite, подключение по URL в DBeaver, схему `public` и горячие клавиши. | Подключаются в DBeaver к своей базе `club_varXX`. |
| **15–35 мин** | **Этап 2. Перевод DDL и конструктор таблиц** | Показывает разницу типов SQLite и PostgreSQL, запускает `demo/01_demo_ddl_postgres.sql`, показывает конструктор таблиц (`Persist Changes`) и вкладку **ER-диаграмма**. | Переводят `sqlite_schema_*.sql` в PostgreSQL, создают 5–6 таблиц и проверяют **ER-диаграмму**. |
| **35–50 мин** | **Этап 3. Импорт CSV, `setval` и вкладка «Данные»** | Показывает импорт CSV через DBeaver и Python (`append`), синхронизацию счетчика `setval` и добавление строки во вкладке **«Данные»**. | Загружают свои CSV, выполняют `setval`, добавляют строку через вкладку **«Данные»**. |
| **50–65 мин** | **Этап 4. Запросы в PostgreSQL (1 таблица)** | По шагам запускает `demo/02_demo_queries_postgres.sql` (`'VIP'`, `ILIKE`, `BOOLEAN`, `COALESCE`, `DATE`, `RETURNING`, `FILTER`, `ROLLBACK`). | Пишут и выполняют 4–5 однотабличных запросов своего уровня. |
| **65–90 мин** | **Этап 5. Самостоятельная работа и приемка** | Проверяет ER-диаграмму, вкладку «Данные» и SQL-скрипт у студентов. | Сдают работу. |

---
## Этап 0 (До начала пары). Быстрая заливка демо-базы `club_demo` на сервер

Выполните в DBeaver под аккаунтом `postgres` команду `CREATE DATABASE club_demo;` (если база еще не создана), впишите IP сервера и пароль в ячейку ниже и запустите её. Скрипт создаст 6 таблиц из `demo/01_demo_ddl_postgres.sql`, загрузит CSV из `demo/variant_00` и настроит счетчики `SERIAL`.

In [ ]:
import os
import pandas as pd
from sqlalchemy import create_engine, text

PG_HOST = os.environ.get("PG_HOST", "81.30.105.12")   # IP сервера
PG_PORT = os.environ.get("PG_PORT", "5432")         # Порт (443 или 5432)
PG_DB   = os.environ.get("PG_DB",   "club_demo")   # Демо-база
PG_USER = os.environ.get("PG_USER", "postgres")    # Логин
PG_PASS = os.environ.get("PG_PASS", "NDS_Password")            # Пароль (впишите для заливки на сервер)

tables_order = ["tariffs", "members", "pcs", "sessions", "bar_orders", "tournament_entries"]

if PG_PASS:
    engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PASS}@{PG_HOST}:{PG_PORT}/{PG_DB}")
    with open(os.path.join("demo", "01_demo_ddl_postgres.sql"), "r", encoding="utf-8") as f:
        ddl_sql = f.read()
    with engine.begin() as conn:
        conn.execute(text(ddl_sql))
    print("[OK] Созданы 6 таблиц в базе club_demo.")

    for tbl in tables_order:
        df_csv = pd.read_csv(os.path.join("demo", "variant_00", f"{tbl}.csv"))
        df_csv.to_sql(tbl, engine, if_exists="append", index=False)
        print(f"  -> {tbl}: загружено {len(df_csv)} строк")

    pk_map = {
        "tariffs": "tariff_id",
        "members": "member_id",
        "pcs": "pc_id",
        "sessions": "session_id",
        "bar_orders": "order_id",
        "tournament_entries": "entry_id",
    }
    with engine.begin() as conn:
        for tbl, pk in pk_map.items():
            val = conn.execute(text(f"SELECT setval(pg_get_serial_sequence('{tbl}', '{pk}'), COALESCE(MAX({pk}), 1)) FROM {tbl};")).scalar()
            print(f"  -> Счетчик {tbl}.{pk} = {val}")
else:
    print("PG_PASS не задан — демонстрационные ячейки ниже работают с локальной копией demo/club_demo_sqlite.db.")

ModuleNotFoundError: No module named 'psycopg2'

---
## Этап 1 (00–15 мин). Архитектура клиент-сервер и подключение в DBeaver

### 1.1. Что кратко пояснить студентам
1. **SQLite — встраиваемая файловая БД:** вся база лежит в одном файле `.db` на диске, нет пользователей, паролей и сети, типы данных не контролируются.
2. **PostgreSQL — клиент-серверная СУБД:** база работает как служба на сервере (VPS или локальной машине), проверяет права доступа (`student_01..20`), строго контролирует типы столбцов и внешние ключи.
3. **Иерархия объектов в PostgreSQL:**
   `Сервер` $\to$ `База данных (club_varXX)` $\to$ `Схема (public)` $\to$ `Таблицы и Последовательности (Sequences)`.

### 1.2. Что показать на экране в DBeaver
1. **Новое соединение $\to$ PostgreSQL $\to$ переключатель URL:**
   `jdbc:postgresql://<IP_СЕРВЕРА>:443/club_varXX`
   *(Логин: `student_XX`, Пароль: `StudyPass2026_varXX`, галочка «Сохранить пароль» $\to$ Тест соединения $\to$ Готово).*
2. **Навигатор БД слева:** раскрыть `club_varXX` $\to$ `Базы данных` $\to$ `club_varXX` $\to$ `Схемы` $\to$ **`public`** $\to$ **`Таблицы`**.
3. **Горячие клавиши SQL-редактора (`F3`):**
   * `Ctrl + Enter` — выполнить один запрос под курсором.
   * `Alt + X` — выполнить весь скрипт целиком.
   * `F5` — обновить папку `Таблицы` или схему `public`.

---
## Этап 2 (15–35 мин). Перевод таблиц (DDL) из SQLite в PostgreSQL и конструктор в GUI

### 2.1. Демонстрация: почему SQLite принимает неверные типы, а PostgreSQL — нет
В SQLite действует динамическая типизация: если объявить `pc_code VARCHAR(10)` и `ram_gb SMALLINT`, туда все равно можно записать строку из 35 символов и текст вместо числа:

In [2]:
import sqlite3
import pandas as pd

conn_mem = sqlite3.connect(":memory:")
conn_mem.execute("CREATE TABLE pcs_sqlite (pc_code VARCHAR(10), ram_gb SMALLINT, is_active BOOLEAN);")
conn_mem.execute("INSERT INTO pcs_sqlite VALUES ('PC-SUPER-MEGA-LONG-CODE-999999999', 'шестнадцать', 'да');")

display(pd.read_sql("SELECT *, typeof(pc_code) AS type_code, typeof(ram_gb) AS type_ram FROM pcs_sqlite", conn_mem))
conn_mem.close()

pc_code,ram_gb,is_active,type_code,type_ram
PC-SUPER-MEGA-LONG-CODE-999999999,шестнадцать,да,text,text


### 2.2. Таблица перевода типов данных: SQLite $\to$ PostgreSQL

| Назначение столбца | В шаблоне SQLite (`sqlite_schema.sql`) | Как переводим в PostgreSQL | Что меняется в PostgreSQL |
| :--- | :--- | :--- | :--- |
| Первичный ключ с автоинкрементом | `INTEGER PRIMARY KEY AUTOINCREMENT` | `SERIAL PRIMARY KEY` | Создает объект-счетчик (`SEQUENCE`), выдающий `1, 2, 3...`. |
| Малые целые числа (часы, ОЗУ) | `INTEGER` | `SMALLINT` | 2 байта (-32768..32767). |
| Обычные целые (внешние ключи) | `INTEGER` | `INTEGER` | 4 байта, запрещает текст и дробные числа. |
| Деньги и цены | `REAL` | `NUMERIC(8, 2)` / `NUMERIC(10, 2)` | Точное десятичное хранение без потери копеек. |
| Строки с ограничением длины | `TEXT` | `VARCHAR(n)` (например, `VARCHAR(10)`) | Строка длиннее `n` символов вызовет ошибку `value too long`. |
| Флаги Да / Нет | `INTEGER DEFAULT 0` (`0`/`1`) | `BOOLEAN DEFAULT FALSE` | Значения `TRUE`/`FALSE`. В DBeaver отображается чекбоксом `[v]`. |
| Дата | `TEXT` (`'2026-02-15'`) | `DATE` (`DEFAULT CURRENT_DATE`) | Календарный тип. В DBeaver открывается календарем. |

### 2.3. Отличия при создании ограничений (`Constraints`)
1. **Внешние ключи (`REFERENCES`):** в PostgreSQL включены всегда (команда `PRAGMA foreign_keys = ON` не нужна).
2. **Именованные ограничения (`CONSTRAINT`):**
   `ram_gb SMALLINT NOT NULL CONSTRAINT chk_pc_ram CHECK (ram_gb IN (16, 32, 64))`
   При нарушении правила PostgreSQL назовет имя `chk_pc_ram` прямо в сообщении об ошибке.
3. **Составной `UNIQUE` (в таблице `tournament_entries`):**
   `CONSTRAINT uq_member_game UNIQUE (member_id, game_title)` — один игрок не может дважды заявиться на одну и ту же игру.
4. **Правила внешнего ключа при удалении (`ON DELETE`):**
   * `ON DELETE RESTRICT` — запрещает удалять родительскую строку, если на неё есть ссылки (например, тариф или ПК).
   * `ON DELETE SET NULL` — при удалении игрока из `members` его сессии в `sessions` остаются, а `member_id` становится `NULL`.
   * `ON DELETE CASCADE` — при удалении игрока все его заявки в `tournament_entries` удаляются автоматически.
5. **Удаление связанных таблиц:** `DROP TABLE IF EXISTS имя_таблицы CASCADE;`.

### 2.4. Показ в GUI DBeaver (Скрипт `demo/01_demo_ddl_postgres.sql`)
1. Откройте в DBeaver файл `demo/01_demo_ddl_postgres.sql` и выполните его (`Alt + X`).
2. **Визуальный конструктор таблиц:** откройте любую таблицу двойным кликом $\to$ вкладка **Свойства** $\to$ **Колонки / Ограничения / Внешние ключи**. Покажите, что при добавлении колонки или ограничения мышкой и нажатии `Ctrl + S` открывается окно **Persist Changes** с готовым кодом `ALTER TABLE` / `CREATE TABLE`.
3. **ER-диаграмма:** нажмите `F5` на схеме `public`, дважды кликните по `public` и перейдите на вкладку **ER-диаграмма** — покажите схему из 6 таблиц и кнопку сохранения диаграммы в `.png`.

---
## Этап 3 (35–50 мин). Загрузка данных из CSV, счетчик `SERIAL` (`setval`) и вкладка «Данные»

### 3.1. Состав таблиц демонстрационного варианта (`demo/variant_00`)

In [3]:
DEMO_DIR = os.path.join("demo", "variant_00")
summary = []
for t in tables_order:
    df_t = pd.read_csv(os.path.join(DEMO_DIR, f"{t}.csv"))
    summary.append({"Таблица": t, "Строк": len(df_t), "Столбцы (4-5 шт.)": ", ".join(df_t.columns)})
display(pd.DataFrame(summary))

Таблица,Строк,Столбцы (4-5 шт.)
tariffs,4,"tariff_id, zone_name, hourly_rate, is_vip"
members,25,"member_id, nickname, balance, is_verified, reg_date"
pcs,12,"pc_id, pc_code, tariff_id, ram_gb, is_active"
sessions,50,"session_id, member_id, pc_id, hours, session_date"
bar_orders,40,"order_id, session_id, item_name, price, is_delivered"
tournament_entries,30,"entry_id, member_id, pc_id, game_title, prize"


### 3.2. Два способа загрузки CSV в PostgreSQL
Загружать таблицы нужно строго от родительских к дочерним: `tariffs` $\to$ `members` $\to$ `pcs` $\to$ `sessions` $\to$ `bar_orders` $\to$ `tournament_entries`.

1. **Через интерфейс DBeaver:** правый клик по таблице $\to$ **Импорт данных** $\to$ **Из файла CSV** $\to$ выбрать файл $\to$ **Приступить**.
2. **Через Python (`if_exists='append'`):**
   ```python
   for tbl in ["tariffs", "members", "pcs", "sessions", "bar_orders"]:
       pd.read_csv(f"easy/variant_01/{tbl}.csv").to_sql(tbl, engine, if_exists="append", index=False)
   ```
   *Важно:* акцент для студентов — использовать только `if_exists="append"`. Режим `"replace"` удалит таблицу со всеми `SERIAL`, `VARCHAR`, `CHECK` и `FOREIGN KEY`.

### 3.3. Почему после импорта CSV нужно выполнить `setval`?
Когда мы загружаем из CSV строки с готовыми `member_id` (от `1` до `25`), внутренний счетчик `SERIAL` в PostgreSQL не двигается и остается равным `1`. При попытке добавить новую строку без указания `member_id` возникнет ошибка:
`ERROR: duplicate key value violates unique constraint "members_pkey" (Key (member_id)=(1) already exists)`.

Решение — синхронизировать счетчик с текущим максимумом таблицы:
```sql
SELECT setval(pg_get_serial_sequence('members', 'member_id'), COALESCE(MAX(member_id), 1)) FROM members;
```

### 3.4. Что показать во вкладке «Данные» в DBeaver
Откройте таблицу `members` или `sessions` $\to$ вкладка **Данные** $\to$ нажмите **`+` (Добавить строку)** внизу:
* Поле `BOOLEAN` переключается **чекбоксом `[v]`**;
* Поле `DATE` выбирается во **всплывающем календаре**;
* При клике на внешний ключ (`member_id` или `pc_id` в `sessions`) справа открывается панель **Словарь (Dictionary)** со списком игроков/ПК из родительской таблицы;
* Нажатие `Ctrl + S` сохраняет строку на сервере.

---
## Этап 4 (50–65 мин). Отличия SQL-запросов PostgreSQL от SQLite (на одной таблице)

Все запросы ниже собраны в файле **`demo/02_demo_queries_postgres.sql`** для пошагового запуска в DBeaver (`Ctrl + Enter`).

### 4.1. Строгий контроль `VARCHAR(10)`, `CHECK` и кавычек (`'VIP'` против `"VIP"`)
1. **Проверка длины `VARCHAR(10)`:**
   `INSERT INTO pcs (pc_code, tariff_id, ram_gb) VALUES ('PC-SUPER-LONG-CODE-99999', 1, 16);`
   $\to$ `ERROR: value too long for type character varying(10)`.
2. **Проверка именованного `CHECK`:**
   `INSERT INTO pcs (pc_code, tariff_id, ram_gb) VALUES ('VIP-99', 3, 24);`
   $\to$ `ERROR: new row for relation "pcs" violates check constraint "chk_pc_ram"`.
3. **Одинарные и двойные кавычки:**
   * В SQLite прощалось `WHERE zone_name = "VIP"`.
   * В PostgreSQL двойные кавычки означают **имя столбца** (`ERROR: column "VIP" does not exist`). Текстовые строки пишутся **только в одинарных кавычках**: `WHERE zone_name = 'VIP'`.

### 4.2. Вывод результатов однотабличных запросов на данных `demo/variant_00`

In [4]:
conn_demo = sqlite3.connect(os.path.join("demo", "club_demo_sqlite.db"))

print("1. Поиск без учета регистра: в PostgreSQL LIKE чувствителен к регистру, поэтому используем ILIKE '%cola%':")
display(pd.read_sql("SELECT order_id, session_id, item_name, price, is_delivered FROM bar_orders WHERE LOWER(item_name) LIKE '%cola%';", conn_demo))

print("2. Фильтр по типу BOOLEAN (в PostgreSQL: WHERE is_active = TRUE AND ram_gb >= 32):")
display(pd.read_sql("SELECT pc_id, pc_code, tariff_id, ram_gb, is_active FROM pcs WHERE is_active = 1 AND ram_gb >= 32 ORDER BY ram_gb DESC, pc_code;", conn_demo))

print("3. Замена NULL через COALESCE (функции IFNULL в PostgreSQL нет):")
display(pd.read_sql("SELECT entry_id, member_id, game_title, prize, COALESCE(prize, 0.00) AS prize_clean FROM tournament_entries ORDER BY prize_clean DESC LIMIT 8;", conn_demo))

print("4. Условная агрегация FILTER (WHERE ...) на одной таблице pcs:")
q_filter = """
SELECT
    COUNT(*) AS "Всего ПК",
    COUNT(*) FILTER (WHERE is_active = 1) AS "Активных ПК",
    COUNT(*) FILTER (WHERE is_active = 0) AS "На ремонте",
    COUNT(*) FILTER (WHERE ram_gb = 64)   AS "ПК 64 ГБ ОЗУ"
FROM pcs;
"""
display(pd.read_sql(q_filter, conn_demo))
conn_demo.close()

1. Поиск без учета регистра: в PostgreSQL LIKE чувствителен к регистру, поэтому используем ILIKE '%cola%':
2. Фильтр по типу BOOLEAN (в PostgreSQL: WHERE is_active = TRUE AND ram_gb >= 32):
3. Замена NULL через COALESCE (функции IFNULL в PostgreSQL нет):
4. Условная агрегация FILTER (WHERE ...) на одной таблице pcs:


order_id,session_id,item_name,price,is_delivered
3,29,Coca-Cola Zero 0.5,280.0,1
7,41,Coca-Cola Zero 0.5,140.0,1
8,16,Coca-Cola Zero 0.5,140.0,1


pc_id,pc_code,tariff_id,ram_gb,is_active
1,BOOT-01,2,32,1
2,BOOT-02,2,32,1
4,BOOT-04,2,32,1
7,BOOT-07,2,32,1
8,BOOT-08,2,32,1
10,BOOT-10,2,32,1
3,VIP-03,3,32,1
12,VIP-12,3,32,1


entry_id,member_id,game_title,prize,prize_clean
3,7,Dota 2,15000.0,15000.0
11,22,League of Legends,15000.0,15000.0
22,4,Valorant,15000.0,15000.0
24,11,Counter-Strike 2,15000.0,15000.0
4,19,Apex Legends,10000.0,10000.0
13,10,Apex Legends,10000.0,10000.0
20,9,Valorant,10000.0,10000.0
30,22,PUBG: Battlegrounds,10000.0,10000.0


Всего ПК,Активных ПК,На ремонте,ПК 64 ГБ ОЗУ
12,11,1,0


### 4.3. Конструкции PostgreSQL для уровней Medium и Hard (показать в DBeaver)

1. **Вычитание дат (`DATE`):**
   ```sql
   SELECT member_id, nickname, reg_date, (CURRENT_DATE - reg_date) AS days_in_club, balance
   FROM members
   WHERE is_verified = TRUE
   ORDER BY reg_date ASC;
   ```
2. **Вставка с возвратом строки (`RETURNING`) и обновлением при дубликате (`ON CONFLICT DO UPDATE`):**
   ```sql
   INSERT INTO members (nickname, balance, is_verified)
   VALUES ('CyberAdmin_Demo', 2500.00, TRUE)
   ON CONFLICT (nickname) DO UPDATE
       SET balance = members.balance + 500.00
   RETURNING member_id, nickname, balance, is_verified, reg_date;
   ```
3. **Безопасная проверка правил `ON DELETE SET NULL` и `ON DELETE CASCADE` через транзакцию (`BEGIN ... ROLLBACK`):**
   ```sql
   BEGIN;
   DELETE FROM members WHERE member_id = 1;
   -- В sessions строки сохранились, а member_id стал NULL:
   SELECT * FROM sessions WHERE member_id IS NULL;
   -- В tournament_entries заявки игрока №1 удалились каскадно:
   SELECT COUNT(*) FROM tournament_entries WHERE member_id = 1;
   -- Откатываем удаление обратно:
   ROLLBACK;
   ```

---
## Этап 5 (65–90 мин). Самостоятельная работа и чек-лист приемки

| Оценка | Уровень | Что быстро проверить у студента в DBeaver |
| :--- | :--- | :--- |
| **«3»** | **Easy (5 таблиц)** | Вкладка **ER-диаграмма** (5 связанных таблиц), добавленная вручную строка в `members` на вкладке **Данные**, 4 запроса в SQL-редакторе (`'VIP'`, `ILIKE`, `BOOLEAN`, тест `VARCHAR(10)`). |
| **«4»** | **Medium (6 таблиц)** | **ER-диаграмма** из 6 таблиц с кольцом связей (`er_diagram_varXX.png`), именованные `CONSTRAINT` и `uq_member_game`, синхронизированный `setval`, 5 запросов (`ILIKE`, `COALESCE`, `CURRENT_DATE - reg_date`, `RETURNING`, тесты `chk_pc_ram` и `uq_member_game`). |
| **«5»** | **Hard (6 таблиц)** | Требования Medium + показ блока `BEGIN ... DELETE ... ROLLBACK` (разница `SET NULL` в `sessions` и `CASCADE` в `tournament_entries`), повторный запуск `UPSERT ... RETURNING`, запросы с `FILTER (WHERE ...)` и выгруженный через GUI DBeaver файл `hard_tournaments_varXX.csv`. |